# WDL — Module 1 · Session 3 — Reading a rain gauge record; time scales
### Reading notebook

| | |
|---|---|
| **Course** | Water Data Literacy: Rainfall, Floods and Climate Change |
| **Edition** | Udemy |
| **Instructor** | Dr Arpan Pradhan |
| **Module** | Module 1 — The Water Cycle as Data |
| **Session** | Session 3 of 30 |
| **Topics** | Rain gauges and the IMD rainfall day; rainy-day definition (2.5 mm); daily, weekly, monthly and seasonal totals; IMD seasons |
| **Water hook** | *IMD's 8:30 am rainfall day* |

---
> **How to use this notebook.** No coding needed. Read each explanation, then press ▶ on the box below it
> (or click the box and press `Shift + Enter`). Some boxes have sliders, number boxes or dropdowns: change them, then press ▶ again.
> Go to **File → Save a copy in Drive** first, so the notebook is yours.

## Why this session matters
Every rainfall figure in the news starts in a **rain gauge**: a funnel over a measuring container. In India the gauge
is read every morning at **8:30 am IST**, recording the rain of the previous 24 hours. In this session you will read a
full year of gauge records and see how the same data tells different stories at different time scales.

In [ ]:
#@title ▶ Step 0 · Get ready (press ▶ once before anything else)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from IPython.display import display, Markdown
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110, "font.size": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, GREY, TEAL = "#2C6E9B", "#DD8452", "#C44E52", "#8C8C8C", "#1F7A8C"
def ok(msg="Correct!"):  display(Markdown(f"✅ **{msg}**"))
def no(msg="Not quite. Check your working and try again."): display(Markdown(f"❌ {msg}"))
def close(a, b, tol=0.02):  # within 2% (or 0.01 for very small numbers)
    try: return abs(float(a) - float(b)) <= max(abs(float(b)) * tol, 0.01)
    except (TypeError, ValueError): return False

DATA_URL = "https://raw.githubusercontent.com/PradhanArpan/waterliteracy/main/data/M1/sample_station_2023.csv"
def _make_sample_station():
    rng = np.random.default_rng(2023)
    days = pd.date_range("2023-01-01", "2023-12-31", freq="D")
    p_wet  = [0.03, 0.03, 0.08, 0.20, 0.35, 0.40, 0.45, 0.48, 0.55, 0.50, 0.30, 0.08]
    mean_r = [3, 3, 6, 9, 10, 7, 7, 8, 11, 11, 9, 4]
    rain = []
    for d in days:
        m = d.month - 1
        rain.append(rng.gamma(0.8, mean_r[m] / 0.8) if rng.random() < p_wet[m] else 0.0)
    rain = np.round(np.array(rain), 1)
    rain[days.get_loc(pd.Timestamp("2023-09-21"))] = 96.4
    rain[days.get_loc(pd.Timestamp("2023-09-22"))] = 58.2
    level, soil, channel = [], 0.0, 0.0
    for r in rain:
        soil = soil * 0.6 + max(r - 5, 0) * 0.5
        channel = channel * 0.82 + soil * 0.022
        level.append(round(1.8 + channel, 2))
    return pd.DataFrame({"date": days, "rain_mm": rain, "level_m": level})
try:
    station = pd.read_csv(DATA_URL, parse_dates=["date"])
except Exception:
    station = _make_sample_station()   # works offline too: same numbers
station = station.set_index("date")
rainfall, river = station[["rain_mm"]], station[["level_m"]]
WARNING_LEVEL, DANGER_LEVEL = 3.6, 4.2
display(Markdown("✅ **Ready.** Sample station loaded: one year (2023) of daily rainfall and river level. "
                 "*Illustrative, synthetic data for practice; not an official IMD or CWC record.*"))

---
## Part 1 · What a rain gauge record looks like
A rainfall record is simply a table: a date and a number. Press ▶.

In [ ]:
#@title ▶ Look at the raw record
display(Markdown("**First 10 days of January** (dry season)"))
display(rainfall.head(10).T)
display(Markdown("**17 to 26 September** (monsoon)"))
display(rainfall.loc["2023-09-17":"2023-09-26"].T)

---
## Part 2 · One year as a chart
A table of 365 numbers hides the pattern. A chart shows it at once.
IMD counts a **rainy day** only when **2.5 mm or more** falls in 24 hours.

In [ ]:
#@title ▶ Plot one year of daily rainfall
r = rainfall.rain_mm
total, rainy, wettest = r.sum(), (r >= 2.5).sum(), r.idxmax()
fig, ax = plt.subplots()
ax.bar(r.index, r, width=1.0, color=BLUE)
ax.annotate(f"Wettest day: {r.max():.0f} mm on {wettest:%d %b}", xy=(wettest, r.max()), xytext=(-190, -8),
            textcoords="offset points", arrowprops=dict(arrowstyle="->", color=GREY))
ax.set_title(f"Daily rainfall, 2023: {total:,.0f} mm in the year, {rainy} rainy days")
ax.set_ylabel("Rainfall (mm/day)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.text(0, -0.15, "Illustrative sample station · synthetic data for practice", transform=ax.transAxes, fontsize=9, color=GREY)
plt.tight_layout(); plt.show()

### Expected output
About **1,065 mm** in the year from **77 rainy days**, with the wettest day (**96 mm**) on **21 September**.

---
## Part 3 · How uneven is rain?
Press ▶ to see what share of the whole year's rain fell on the wettest few days.

In [ ]:
#@title ▶ Share of the year's rain on the wettest days { run: "auto" }
top_n_days = 10 #@param {type:"slider", min:1, max:60, step:1}
r = rainfall.rain_mm.sort_values(ascending=False)
share = r.head(top_n_days).sum() / r.sum() * 100
display(Markdown(f"The **{top_n_days} wettest days** brought **{share:.0f}%** of the year's rain "
                 f"({top_n_days/365*100:.1f}% of the days in the year)."))

Rain that arrives in a few big bursts is harder to store and more likely to flood than the same amount spread out.
**The yearly total alone never tells the full story.**

---
## Part 4 · Same data, different time scales
Water data can be added up by day, week, month or season. Choose a scale and press ▶. Try all four.

In [ ]:
#@title ▶ Change the time scale { run: "auto" }
time_scale = "Monthly" #@param ["Daily", "Weekly", "Monthly", "Seasonal"]
r = rainfall.rain_mm
fig, ax = plt.subplots()
if time_scale == "Seasonal":
    season = {1: "Winter\n(Jan–Feb)", 2: "Winter\n(Jan–Feb)", 3: "Pre-monsoon\n(Mar–May)", 4: "Pre-monsoon\n(Mar–May)",
              5: "Pre-monsoon\n(Mar–May)", 6: "SW monsoon\n(Jun–Sep)", 7: "SW monsoon\n(Jun–Sep)", 8: "SW monsoon\n(Jun–Sep)",
              9: "SW monsoon\n(Jun–Sep)", 10: "NE monsoon\n(Oct–Dec)", 11: "NE monsoon\n(Oct–Dec)", 12: "NE monsoon\n(Oct–Dec)"}
    order = ["Winter\n(Jan–Feb)", "Pre-monsoon\n(Mar–May)", "SW monsoon\n(Jun–Sep)", "NE monsoon\n(Oct–Dec)"]
    s = r.groupby(r.index.month.map(season)).sum().reindex(order)
    ax.bar_label(ax.bar(s.index, s.values, color=BLUE, width=0.6), fmt="%.0f mm")
    ax.set_title("Seasonal rainfall (IMD seasons)")
else:
    rule = {"Daily": "D", "Weekly": "W", "Monthly": "MS"}[time_scale]
    s = r.resample(rule).sum()
    b = ax.bar(s.index, s.values, width={"D": 1, "W": 5, "MS": 22}[rule], color=BLUE, align="edge" if rule == "MS" else "center")
    if rule == "MS": ax.bar_label(b, fmt="%.0f", fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.set_title(f"{time_scale} rainfall totals")
ax.set_ylabel("Rainfall (mm)")
plt.tight_layout(); plt.show()
display(Markdown(f"The year's total is the same in every view: **{r.sum():,.0f} mm**. Only the lens changes."))

### 🤔 Think about it
1. Which scale shows a flood-causing storm best? Which suits a farmer planning a crop?
2. In the monthly view, where did the 96 mm storm go?
3. This station gets a large share of its rain from October to December. Why might "monsoon" mean different months in different parts of India?

### ⏭️ Next
Open the **Practice activity** and answer five questions from this record.